[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C34_Agent_Orchestration_Course/01_subagents/01_subagents.ipynb)

# 01 · 子 Agent（Subagents）

目标：用**纯标准库**从零写出 subagent 这个零件——**派生 spawn → 上下文隔离 → 结构化结果回传 → 并行扇出 fan-out → 失败隔离 → 成本聚合**，全程用 **MockLLM** 当每个 agent 的大脑、`assert` 验证，**无需 API key**。

路线：MockLLM → spawn 一个 subagent → 验证上下文隔离 → 结构化回传契约 → fan-out + gather → 失败/超时隔离 → 成本预算 → ✏️ 练习 → 📖 答案 → 🧪 真实 orchestrator-worker 形状胶囊。

> 心智模型：**subagent = 父 agent 雇来的、有独立工作台（隔离上下文）的临时工**——派工单、独立干、只交结论。难点在隔离要真隔离、结果要可对齐汇总、单个失败别连累全局。

## 1 · MockLLM：每个 agent 的确定性大脑

先把本课的 MockLLM 落地（与模块 00 同款，略增强）：规则驱动、确定性、记录调用数与 token（供后面成本聚合）。

subagent 里的 LLM 调用都走它——能力不确定性被剥离，剩下全是我们的编排逻辑。

In [ ]:
import json, time

class MockLLM:
    '''确定性假模型：按 [(关键词, 响应)] 规则映射 prompt -> 响应。
       记录 calls 与 total_tokens(用字符数粗略模拟)，供成本断言。'''
    def __init__(self, rules, default='(no rule)'):
        self.rules, self.default = rules, default
        self.calls, self.total_tokens = 0, 0
    def __call__(self, prompt):
        self.calls += 1
        text = prompt if isinstance(prompt, str) else json.dumps(prompt, ensure_ascii=False)
        self.total_tokens += len(text)
        for kw, resp in self.rules:
            if kw in text:
                self.total_tokens += len(resp)
                return resp
        self.total_tokens += len(self.default)
        return self.default

llm = MockLLM(rules=[('A公司', 'A公司营收 +10%'), ('B公司', 'B公司营收 +12%'),
                     ('C公司', 'C公司营收 -3%')])
print(llm('调研 A公司 财报'), '|', llm('调研 B公司 财报'))
assert llm('看看 C公司') == 'C公司营收 -3%'
assert llm.calls == 3 and llm.total_tokens > 0
print('✅ MockLLM 就绪：确定性、记录调用数与 token')

## 2 · 派生一个 subagent（spawn）

subagent 派生时带齐四要素：**任务说明 + 隔离上下文 + 工具子集 + 结果契约**。
最小实现：`run_subagent(id, task, llm, tools)` 内部**新建独立 messages**、跑一个（可多步）回路、返回结构化结果。

In [ ]:
def run_subagent(subagent_id, task, llm, tools=None):
    '''一个最小 subagent：独立上下文 + 跑回路 + 结构化回传。
       返回契约: {subagent_id, task, status, result, tokens, n_steps}。'''
    messages = []                                  # ← 每次都新建：隔离上下文的根本
    tools = tools or []
    messages.append({'role': 'user', 'content': task})
    tok_before = llm.total_tokens
    answer = llm(task)                             # 真实里: llm(messages) 可能多步
    messages.append({'role': 'assistant', 'content': answer})
    return {'subagent_id': subagent_id, 'task': task, 'status': 'ok',
            'result': answer, 'tokens': llm.total_tokens - tok_before,
            'n_steps': 1, 'ctx': messages}          # ctx 仅为演示隔离，真实不回传

TOOLS_READONLY = ['web_search', 'read_file']       # 检索 subagent 的工具子集(最小权限)
r = run_subagent('researcher-A', '调研 A公司 财报', llm, tools=TOOLS_READONLY)
print('回传契约:', {k: v for k, v in r.items() if k != 'ctx'})
assert set(r) >= {'subagent_id', 'task', 'status', 'result', 'tokens', 'n_steps'}
assert r['status'] == 'ok' and r['result'] == 'A公司营收 +10%'
assert r['tokens'] > 0
print('✅ 派生 subagent：四要素齐全，回传结构化结果（带成本字段）')

## 3 · 上下文隔离：验证不变量

subagent 的命脉是**隔离上下文**：对一个的修改不影响另一个。
派两个 subagent，断言：① 两者上下文是**不同对象**；② 一个的内容**不出现**在另一个里；③ 改一个不影响另一个。

In [ ]:
ra = run_subagent('A', '调研 A公司 财报', MockLLM([('A公司', 'A公司 +10%')]))
rb = run_subagent('B', '调研 B公司 财报', MockLLM([('B公司', 'B公司 +12%')]))

ctx_a, ctx_b = ra['ctx'], rb['ctx']
# ① 不同对象
assert ctx_a is not ctx_b, '两个 subagent 的上下文必须是不同对象'
# ② 互不串味：A 的任务/结果不在 B 的上下文里
b_text = json.dumps(ctx_b, ensure_ascii=False)
assert 'A公司' not in b_text and ra['result'] not in b_text
# ③ 改一个不影响另一个
ctx_a.append({'role': 'user', 'content': '追加给 A 的内容'})
assert len(ctx_a) == 3 and len(ctx_b) == 2, '改 A 的上下文不应影响 B'
print('A 上下文长度', len(ctx_a), '| B 上下文长度', len(ctx_b))
print('✅ 上下文隔离成立：不同对象、互不串味、改一个不影响另一个')

# 反例警示：可变默认参数会破坏隔离(所有调用共享同一个 list)
def BAD_subagent(task, _shared=[]):          # ← 千万别这样！
    _shared.append(task); return list(_shared)
h1 = BAD_subagent('任务1'); h2 = BAD_subagent('任务2')
assert h2 == ['任务1', '任务2'], '可变默认参数 -> 历史串味(隔离失效)'
print('⚠️ 反例：可变默认参数 def f(x=[]) 会让所有 subagent 共享历史 ->', h2)

## 4 · 结果回传契约：只交结论 + 状态 + 成本

回传**结构化结果**而非整段对话。父 agent 据 `status` 过滤、据 `result` 合成、据 `tokens` 聚合成本。
写一个校验器确认每个 subagent 的回传**符合契约**——这是父 agent 能程序化汇总的前提。

In [ ]:
REQUIRED_FIELDS = {'subagent_id', 'task', 'status', 'result', 'tokens', 'n_steps'}
VALID_STATUS = {'ok', 'failed', 'partial', 'timeout'}

def validate_result(r):
    '''校验 subagent 回传是否符合契约。返回 (ok, 错误信息或None)。'''
    if not isinstance(r, dict):
        return False, '结果必须是 dict'
    missing = REQUIRED_FIELDS - set(r)
    if missing:
        return False, f'缺少字段: {sorted(missing)}'
    if r['status'] not in VALID_STATUS:
        return False, f"status 非法: {r['status']}"
    if not isinstance(r['tokens'], int) or r['tokens'] < 0:
        return False, 'tokens 必须是非负整数'
    return True, None

good = {'subagent_id': 'r1', 'task': 't', 'status': 'ok', 'result': '...',
        'tokens': 100, 'n_steps': 3}
print(validate_result(good))
assert validate_result(good) == (True, None)
assert validate_result({'subagent_id': 'r1'})[0] is False        # 缺字段
assert validate_result({**good, 'status': 'weird'})[0] is False   # 非法状态
assert validate_result({**good, 'tokens': -1})[0] is False        # 负 token
# 我们前面 run_subagent 的产物去掉 ctx 后应符合契约
r_clean = {k: v for k, v in run_subagent('x', '调研 A公司', llm).items() if k != 'ctx'}
assert validate_result(r_clean)[0] is True
print('✅ 回传契约校验：合法放行、缺字段/非法状态/负成本全部拦下')

## 5 · 并行扇出 fan-out + 扇入 gather

子任务互不依赖时，**同时**派多个 subagent（本课顺序模拟并发，逻辑相同），再 **gather** 收集。
两条铁律：① 结果数 == 子任务数；② 靠 `subagent_id` 对齐（即使乱序）。

In [ ]:
def fan_out(subtasks, llm, tools=None):
    '''对每个子任务派一个 subagent(顺序模拟并发)，gather 全部结果。'''
    results = []
    for i, st in enumerate(subtasks):
        sid = f'sub-{i}'
        r = {k: v for k, v in run_subagent(sid, st, llm, tools).items() if k != 'ctx'}
        results.append(r)
    return results

def gather_by_id(results):
    '''按 subagent_id 建索引，便于父 agent 对号入座地汇总(即使结果乱序)。'''
    return {r['subagent_id']: r for r in results}

company_llm = MockLLM(rules=[('A公司', 'A +10%'), ('B公司', 'B +12%'), ('C公司', 'C -3%')])
subtasks = ['调研 A公司', '调研 B公司', '调研 C公司']
results = fan_out(subtasks, company_llm, tools=TOOLS_READONLY)
for r in results:
    print(r['subagent_id'], '->', r['result'], f"({r['tokens']} tok)")
# 铁律1: 结果数 == 子任务数
assert len(results) == len(subtasks)
# 铁律2: 按 id 对齐
idx = gather_by_id(results)
assert idx['sub-1']['result'] == 'B +12%'      # 第 2 个子任务=B公司
assert [r['subagent_id'] for r in results] == ['sub-0', 'sub-1', 'sub-2']
print('✅ fan-out + gather：3 个子任务 -> 3 个结果，按 id 可对齐汇总')

## 6 · 失败/超时隔离 + 成本聚合

一个 subagent 崩了，**别拖垮全局**：本地捕获 → 回传一个 `failed` 结果（结果数仍不变）。
再给每个 subagent **步数上限**模拟超时。最后**聚合**所有 subagent 的 token = 一次运行的总成本（模块 04 会系统化）。

In [ ]:
def run_subagent_safe(subagent_id, task, run_one, max_steps=5):
    '''带失败隔离 + 步数上限的 subagent 包装。run_one(task)->(result, steps)。'''
    try:
        result, steps = run_one(task)
        if steps > max_steps:
            return {'subagent_id': subagent_id, 'task': task, 'status': 'timeout',
                    'result': None, 'tokens': 0, 'n_steps': max_steps}
        return {'subagent_id': subagent_id, 'task': task, 'status': 'ok',
                'result': result, 'tokens': len(str(result)), 'n_steps': steps}
    except Exception as e:
        return {'subagent_id': subagent_id, 'task': task, 'status': 'failed',
                'result': None, 'tokens': 0, 'n_steps': 0,
                'error': f'{type(e).__name__}: {e}'}

def worker(task):
    if task == 'BOOM':   raise RuntimeError('子任务崩溃')
    if task == 'SLOW':   return 'slow-result', 99       # 步数超限 -> timeout
    return task + '-done', 2

tasks = ['t1', 'BOOM', 'SLOW', 't2']
res = [run_subagent_safe(f's{i}', t, worker) for i, t in enumerate(tasks)]
for r in res:
    print(f"{r['subagent_id']:4s} {r['status']:8s} result={r['result']}")
# 结果数不变，状态各异
assert len(res) == 4
assert [r['status'] for r in res] == ['ok', 'failed', 'timeout', 'ok']
# 成本聚合：只算成功完成的 token；并统计成功率
total_tokens = sum(r['tokens'] for r in res)
ok_rate = sum(r['status'] == 'ok' for r in res) / len(res)
print(f'总 token={total_tokens} | 成功率={ok_rate:.2f}')
assert ok_rate == 0.5 and total_tokens == len('t1-done') + len('t2-done')
print('✅ 失败/超时被隔离成结果、结果数不变；总成本可聚合')

---
## ✏️ 练习 1：带 token 预算的 subagent

多 agent 很费 token，要给每个 subagent **预算上限**。

实现 `run_with_budget(task, cost_fn, budget)`：`cost_fn(task)` 返回这个子任务**预计花费的 token**；若预计花费 > budget，**不执行**，返回 `{'status':'rejected', 'reason':'over budget', 'tokens':0}`；否则返回 `{'status':'ok', 'result':task.upper(), 'tokens':cost_fn(task)}`。

In [ ]:
def run_with_budget(task, cost_fn, budget):
    # TODO: cost = cost_fn(task)
    #   若 cost > budget -> {'status':'rejected','reason':'over budget','tokens':0}
    #   否则           -> {'status':'ok','result':task.upper(),'tokens':cost}
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
cost = lambda t: len(t) * 10
r_ok = run_with_budget('abc', cost, budget=100)      # 30 <= 100 -> ok
r_no = run_with_budget('abcdefghij', cost, budget=50) # 100 > 50 -> rejected
assert r_ok['status'] == 'ok' and r_ok['result'] == 'ABC' and r_ok['tokens'] == 30
assert r_no['status'] == 'rejected' and r_no['tokens'] == 0
print('预算内:', r_ok, '\n超预算:', r_no)
print('✅ 练习 1 通过：超预算的 subagent 被拒绝执行，省下 token')

## ✏️ 练习 2：扇入对齐——把结果合并回子任务顺序

fan-out 的结果可能**乱序**返回（真实并发里常见）。父 agent 要按**原子任务顺序**重排结果。

实现 `align_results(subtasks, results)`：`results` 是乱序的回传列表（每个含 `task` 字段），返回一个与 `subtasks` **同序**的结果列表；某子任务若无对应结果，则填 `{'task':该子任务,'status':'missing'}`。

In [ ]:
def align_results(subtasks, results):
    # TODO: 按 result['task'] 建索引，再按 subtasks 顺序取出；
    #       缺失的填 {'task': st, 'status': 'missing'}
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
subs = ['查A', '查B', '查C']
shuffled = [{'task': '查C', 'status': 'ok', 'result': 'C'},
            {'task': '查A', 'status': 'ok', 'result': 'A'}]   # B 缺失、且乱序
aligned = align_results(subs, shuffled)
assert [r['task'] for r in aligned] == subs                # 恢复原序
assert aligned[0]['result'] == 'A' and aligned[2]['result'] == 'C'
assert aligned[1]['status'] == 'missing'                   # B 标记缺失
print('对齐后:', [(r['task'], r['status']) for r in aligned])
print('✅ 练习 2 通过：乱序结果按子任务顺序重排，缺失项被标记')

## ✏️ 练习 3：合成——把成功的 subagent 结果汇成最终答案

orchestrator 的收尾是**合成**：只用 `status=='ok'` 的结果，拼成最终答案，并报告完成度。

实现 `synthesize(results)`：返回 `{'answer': '；'.join(成功结果的 result), 'n_ok': 成功数, 'n_total': 总数, 'complete': 是否全部成功}`。

In [ ]:
def synthesize(results):
    # TODO: ok = [r for r in results if r['status']=='ok']
    #   answer = '；'.join(str(r['result']) for r in ok)
    #   返回 {'answer':answer, 'n_ok':len(ok), 'n_total':len(results), 'complete':len(ok)==len(results)}
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
rs = [{'status': 'ok', 'result': 'A +10%'},
      {'status': 'failed', 'result': None},
      {'status': 'ok', 'result': 'C -3%'}]
out = synthesize(rs)
assert out['answer'] == 'A +10%；C -3%'      # 只合成成功的
assert out['n_ok'] == 2 and out['n_total'] == 3 and out['complete'] is False
all_ok = synthesize([{'status': 'ok', 'result': 'x'}])
assert all_ok['complete'] is True
print('合成结果:', out)
print('✅ 练习 3 通过：只合成成功结果、报告完成度（不完整也能给答案）')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def run_with_budget(task, cost_fn, budget):
    cost = cost_fn(task)
    if cost > budget:
        return {'status': 'rejected', 'reason': 'over budget', 'tokens': 0}
    return {'status': 'ok', 'result': task.upper(), 'tokens': cost}

In [ ]:
# 练习 2 参考答案
def align_results(subtasks, results):
    by_task = {r['task']: r for r in results}
    return [by_task.get(st, {'task': st, 'status': 'missing'}) for st in subtasks]

In [ ]:
# 练习 3 参考答案
def synthesize(results):
    ok = [r for r in results if r['status'] == 'ok']
    return {'answer': '；'.join(str(r['result']) for r in ok),
            'n_ok': len(ok), 'n_total': len(results),
            'complete': len(ok) == len(results)}

---
## 🧪 真实数据胶囊：Anthropic multi-agent research system 的 orchestrator-worker 形状

Anthropic《How we built our multi-agent research system》里，一次研究的真实形状是：**lead agent 分解 → 派生 N 个 subagent 并行检索 → 各自隔离上下文 → 回传结构化结论 → lead 合成**。

下面用**贴近真实**的子任务与回传形状，端到端跑一遍这个结构（仍用 MockLLM 保持确定）。

> 形状对照：真实里每个 subagent 是一次独立的 `messages.create(model='claude-opus-4-8', ...)` 会话；本课用 `run_subagent` 模拟，回传字段（result / tokens / sources）贴近真实。

In [ ]:
# 贴近真实的研究任务：调研三家公司的最新财报并对比
RESEARCH_TASK = '对比 A、B、C 三家公司最新财报'
research_llm = MockLLM(rules=[
    # 合成规则放最前、用独特触发词，避免被子结果里的『A公司』等先命中(规则按序匹配)
    ('SYNTH', '综合：B 增长与盈利双优，A 次之，C 承压'),   # lead 合成用
    ('A公司', 'A公司 Q3 营收同比 +10%，净利率 15%'),
    ('B公司', 'B公司 Q3 营收同比 +12%，净利率 18%'),
    ('C公司', 'C公司 Q3 营收同比 -3%，净利率 9%'),
])

def lead_decompose(task):
    '''lead agent 分解：把对比任务拆成对每家公司的独立检索子任务。'''
    return [f'调研 {co}公司 最新财报' for co in ['A', 'B', 'C']]

def lead_synthesize(task, results):
    '''lead agent 合成：把各 subagent 结论喂回 LLM 得到最终对比。'''
    bullets = '\n'.join(f"- {r['result']}" for r in results if r['status'] == 'ok')
    final = research_llm(f'SYNTH 生成综合对比:\n{bullets}')   # SYNTH 触发合成规则
    return {'final': final, 'evidence': [r['result'] for r in results if r['status'] == 'ok'],
            'total_tokens': sum(r['tokens'] for r in results)}

# 端到端：分解 -> fan-out -> 合成
subtasks = lead_decompose(RESEARCH_TASK)
results = fan_out(subtasks, research_llm, tools=TOOLS_READONLY)
report = lead_synthesize(RESEARCH_TASK, results)
print('子任务数:', len(subtasks))
print('最终对比:', report['final'])
print('证据条数:', len(report['evidence']), '| 总 token:', report['total_tokens'])
assert len(subtasks) == 3 and len(results) == 3
assert report['final'].startswith('综合')
assert len(report['evidence']) == 3 and report['total_tokens'] > 0
print('✅ 复现 orchestrator-worker：lead 分解 -> subagent 并行检索(隔离) -> lead 合成')

**🧪 胶囊练习**：实现 `cheapest_subagent(results)`：给定一次 fan-out 的回传列表，返回**消耗 token 最少**的那个 subagent 的 `subagent_id`。（运维里常这样找『哪个 subagent 最省 / 最费』来优化分解粒度。）

In [ ]:
def cheapest_subagent(results):
    # TODO: 返回 tokens 最小的结果的 subagent_id
    raise NotImplementedError

In [ ]:
# 自测
rs = [{'subagent_id': 'a', 'tokens': 50}, {'subagent_id': 'b', 'tokens': 20},
      {'subagent_id': 'c', 'tokens': 80}]
assert cheapest_subagent(rs) == 'b'
print('最省的 subagent:', cheapest_subagent(rs))
print('✅ 胶囊练习通过')

In [ ]:
# 📖 胶囊参考答案
def cheapest_subagent(results):
    return min(results, key=lambda r: r['tokens'])['subagent_id']

---
## 🔧 旁注：对应的真实 Claude 调用（subagent = 一次独立会话）

本课用 `run_subagent` 模拟的「派生一个有隔离上下文的 subagent」，换成真实 Claude 只是**为每个 subagent 开一次独立的 `messages.create` 会话**（伪代码，**本环境不跑、需 API key；无 key 自动回退 MockLLM**）：

```python
import anthropic
client = anthropic.Anthropic()                       # 读 ANTHROPIC_API_KEY

def run_subagent_real(subagent_id, task, tools):
    messages = [{'role': 'user', 'content': task}]    # ← 每个 subagent 独立的 messages = 隔离上下文
    resp = client.messages.create(
        model='claude-opus-4-8', max_tokens=1024,
        tools=tools,                                  # 工具子集(最小权限)
        messages=messages)
    text = ''.join(b.text for b in resp.content if b.type == 'text')
    return {'subagent_id': subagent_id, 'task': task, 'status': 'ok',
            'result': text,
            'tokens': resp.usage.input_tokens + resp.usage.output_tokens}  # 真实 token 数!

# fan-out 真实并发: 用线程池 / asyncio 同时发起多个 run_subagent_real
# from concurrent.futures import ThreadPoolExecutor
# with ThreadPoolExecutor() as ex:
#     results = list(ex.map(lambda st: run_subagent_real(st[0], st[1], TOOLS), subtasks))
```

对应关系：`run_subagent` ↔ 一次独立 `messages.create`、独立 messages ↔ 隔离上下文、`resp.usage` ↔ 真实 token、我们的 fan-out / gather / 对齐 / 失败隔离 / 合成逻辑**原样适用**（顺序模拟 → 换成线程池即真并发）。这就是「scaffold 可迁移」。

### 小结
- subagent = 父 agent 雇来的、有**独立工作台（隔离上下文）**的临时工：派工单、独立干、只交结论。
- **四要素**：任务说明 + 隔离上下文 + 工具子集 + 结果契约。
- **隔离要真隔离**：各建可变状态，绝不共享同一个 list/dict（小心可变默认参数）。
- **回传只交结论 + 状态 + 成本**，不交整段对话——这是父 agent 上下文不爆、能程序化汇总的前提。
- **fan-out 铁律**：结果数 == 子任务数（部分失败也一个不少）、按 id 对齐。
- **失败/超时隔离**：单个 subagent 干净地失败成一个结果，绝不炸到全局；总成本可聚合。

下一站：**模块 02 · 编排模式** —— 把 subagent 用 pipeline / fan-out / router / supervisor-worker 四种拓扑组织起来。